# 06 — Strengths, Limitations, and Complexity Experiments

    **Companion chapter:** `06-strengths-and-limitations.md`

    ## Learning goals

    - Represent model comparisons programmatically.
- Visualize recurrent and self-attention path lengths.
- Estimate the quadratic memory cost of attention scores.
- Distinguish parallel training from sequential generation.
- Translate trade-offs into explicit, critiqueable assumptions.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 1. Encode the architecture comparison as data

Turning conceptual comparisons into a dataframe makes them easier to expand,
sort, visualize, and maintain in a repository.

In [2]:
comparison = pd.DataFrame(
    [
        ["n-gram", "Fixed n-1 history", True, "Sparse counts and short context"],
        ["Feed-forward neural LM", "Learned fixed window", True, "Fixed context"],
        ["RNN", "Recurrent hidden state", False, "Long recurrent paths"],
        ["LSTM/GRU", "Gated recurrent memory", False, "Still sequential"],
        ["RNN + attention", "Dynamic decoder context", False, "Recurrence remains"],
        ["Transformer", "Self- and cross-attention", True, "Quadratic attention"],
    ],
    columns=["model", "context mechanism", "parallel over tokens", "main bottleneck"],
)
comparison

,model,context mechanism,parallel over tokens,main bottleneck
0,n-gram,Fixed n-1 history,True,Sparse counts and short context
1,Feed-forward neural LM,Learned fixed window,True,Fixed context
2,RNN,Recurrent hidden state,False,Long recurrent paths
3,LSTM/GRU,Gated recurrent memory,False,Still sequential
4,RNN + attention,Dynamic decoder context,False,Recurrence remains
5,Transformer,Self- and cross-attention,True,Quadratic attention


## 2. Compare dependency path lengths

In an RNN, information may pass through approximately `distance` recurrent
transitions. In one full self-attention layer, two positions can interact directly.

In [3]:
distances = np.arange(1, 101)
rnn_path = distances
self_attention_path = np.ones_like(distances)

plt.plot(distances, rnn_path, label="RNN")
plt.plot(distances, self_attention_path, label="Self-attention")
plt.xlabel("Token distance")
plt.ylabel("Shortest conceptual path length")
plt.title("Path length between distant positions")
plt.legend()
plt.show()

/tmp/ipykernel_852/1416562623.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Estimate attention-score memory

This estimates only the attention-score tensors, not embeddings, gradients,
optimizer states, Q/K/V tensors, or framework overhead.

In [4]:
def attention_score_memory_mb(
    sequence_length: int,
    num_heads: int = 8,
    batch_size: int = 1,
    bytes_per_value: int = 4,
) -> float:
    values = batch_size * num_heads * sequence_length * sequence_length
    return values * bytes_per_value / (1024 ** 2)


lengths = [128, 512, 1_024, 4_096, 16_384]
memory_table = pd.DataFrame(
    {
        "sequence length": lengths,
        "score pairs per head": [length ** 2 for length in lengths],
        "score memory MB (8 heads, fp32)": [
            attention_score_memory_mb(length) for length in lengths
        ],
        "score memory MB (8 heads, fp16/bf16)": [
            attention_score_memory_mb(length, bytes_per_value=2)
            for length in lengths
        ],
    }
)
memory_table

,sequence length,score pairs per head,"score memory MB (8 heads, fp32)","score memory MB (8 heads, fp16/bf16)"
0,128,16384,0.5,0.25
1,512,262144,8.0,4.00
2,1024,1048576,32.0,16.00
3,4096,16777216,512.0,256.00
4,16384,268435456,8192.0,4096.00


In [5]:
plt.plot(
    memory_table["sequence length"],
    memory_table["score memory MB (8 heads, fp32)"],
    marker="o",
)
plt.xscale("log", base=2)
plt.yscale("log", base=2)
plt.xlabel("Sequence length")
plt.ylabel("Approximate score memory (MB)")
plt.title("Quadratic growth of standard self-attention")
plt.show()

/tmp/ipykernel_852/529669123.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Materialize small score matrices

We avoid very large allocations. The point is to connect the theoretical
`n × n` shape with actual tensor storage.

In [6]:
rows = []
for length in [64, 128, 256, 512, 1024]:
    scores = torch.empty(length, length, dtype=torch.float32)
    rows.append(
        {
            "length": length,
            "shape": str(tuple(scores.shape)),
            "values": scores.numel(),
            "actual MB": scores.numel() * scores.element_size() / (1024 ** 2),
        }
    )

pd.DataFrame(rows)

,length,shape,values,actual MB
0,64,"(64, 64)",4096,0.015625
1,128,"(128, 128)",16384,0.062500
2,256,"(256, 256)",65536,0.250000
3,512,"(512, 512)",262144,1.000000
4,1024,"(1024, 1024)",1048576,4.000000


## 5. Autoregressive generation remains sequential

Parallel training does not imply parallel ordinary generation. This small loop
makes the dependency explicit.

In [7]:
generated = ["<BOS>"]
toy_next_token = {
    "<BOS>": "من",
    "من": "کتاب",
    "کتاب": "می",
    "می": "خوانم",
    "خوانم": "<EOS>",
}

while generated[-1] != "<EOS>":
    generated.append(toy_next_token[generated[-1]])

print(" → ".join(generated))

<BOS> → من → کتاب → می → خوانم → <EOS>


## 6. A transparent teaching heuristic

This is not a universal model-selection rule. It simply converts chapter-level
trade-offs into executable decision logic that students can critique.

In [8]:
def teaching_recommendation(
    sequence_length: int,
    dataset_size: str,
    need_parallel_training: bool,
) -> str:
    if sequence_length <= 5 and dataset_size == "small":
        return "Start with an n-gram or small feed-forward baseline."
    if dataset_size == "small" and not need_parallel_training:
        return "Try an LSTM/GRU baseline before a larger Transformer."
    if sequence_length > 4096:
        return "Investigate long-context or sparse-attention methods."
    return "A small Transformer is a reasonable experimental baseline."


examples = [
    (4, "small", True),
    (100, "small", False),
    (512, "large", True),
    (10_000, "large", True),
]

for example in examples:
    print(example, "->", teaching_recommendation(*example))

(4, 'small', True) -> Start with an n-gram or small feed-forward baseline.
(100, 'small', False) -> Try an LSTM/GRU baseline before a larger Transformer.
(512, 'large', True) -> A small Transformer is a reasonable experimental baseline.
(10000, 'large', True) -> Investigate long-context or sparse-attention methods.


## Practice

1. Add gradient and optimizer-state multipliers to the memory estimate.
2. Compare fp32, fp16, and bf16 score storage.
3. Add a hypothetical local-attention cost of `n × window`.
4. Explain why the score matrix is quadratic but an FFN is position-wise.
5. Critique the teaching heuristic and replace it with your own assumptions.